# Savant 06 train on self-play conversations

This notebook trains a LoRA adapter on the conversations that 05-selfplay kept. Only assistant turns that worked carry labels, so failed code and rejected answers are context the model reads but never learns to write. Adapters are saved at four points across two epochs, and 07-eval picks among them by task results on separate development tables.


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import json
import random
import subprocess
import sys
import time
from pathlib import Path

build = sorted(glob.glob("/kaggle/input/**/BUILD", recursive=True))
assert build and Path(build[0]).read_text().strip() == "EXPECTED_BUILD", "stale bench dataset, stop"
wheel = sorted(glob.glob("/kaggle/input/**/savant-*.whl", recursive=True))[-1]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel], check=True)
# The Kaggle image ships an old torchao that peft refuses to load beside.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=False)
try:
    import peft  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
import torch  # noqa: E402

SOURCE = sorted(glob.glob("/kaggle/input/**/trajectories.jsonl", recursive=True))
assert SOURCE, "no trajectories from 05-selfplay"
print(SOURCE[0], [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Settings for this run.
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MAX_LEN = 6144  # longest conversation in tokens, longer ones are skipped
EPOCHS = 2
LR = 5e-5
ACCUM = 8  # conversations per optimizer step
LORA_R, LORA_ALPHA = 16, 32
HELD_OUT_SHARE = 0.1  # tables kept aside to report loss, never used to choose a checkpoint
SAVE_AT = [0.5, 1.0, 1.5, 2.0]  # epochs at which an adapter is saved
OUT = Path("/kaggle/working/adapters")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from savant.finetune import IGNORE, encode_example

tok = AutoTokenizer.from_pretrained(MODEL_ID)


def render(messages, add_generation_prompt=False):
    return tok.apply_chat_template(
        messages, add_generation_prompt=add_generation_prompt, tokenize=False, enable_thinking=False
    )


def encode(text):
    return tok(text, add_special_tokens=False)["input_ids"]


rows = [json.loads(line) for line in Path(SOURCE[0]).read_text().splitlines()]
tables = sorted({r["task"] for r in rows})
random.Random(0).shuffle(tables)
held_tables = set(tables[: max(1, int(len(tables) * HELD_OUT_SHARE))])
train, held, skipped = [], [], {"too_long": 0, "template": 0, "nothing_to_learn": 0}
for r in rows:
    if not any(r["learn"]):
        skipped["nothing_to_learn"] += 1
        continue
    try:
        ex = encode_example(r["messages"], render, encode, MAX_LEN, learn=r["learn"])
    except ValueError:
        skipped["template"] += 1
        continue
    if ex is None:
        skipped["too_long"] += 1
        continue
    (held if r["task"] in held_tables else train).append(ex)
random.Random(1).shuffle(train)
lengths = sorted(len(ids) for ids, _ in train + held)
print(len(rows), "rows,", len(train), "train,", len(held), "held out,", skipped)
print("median tokens", lengths[len(lengths) // 2], "max", lengths[-1])
assert len(train) >= 100, "too few conversations to train on"

In [ ]:
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()  # adapter weights train in full precision
model.print_trainable_parameters()
base = model.get_base_model()
first = next(base.parameters()).device


def loss_of(ids, labels):
    # Logits only where a label exists, which saves gigabytes on long conversations.
    x = torch.tensor([ids], device=first)
    y = torch.tensor(labels)
    pos = torch.nonzero(y[1:] != IGNORE).squeeze(1)
    with torch.autocast("cuda", dtype=torch.float16):
        h = base.model(input_ids=x).last_hidden_state[0]
        logits = base.lm_head(h[pos.to(h.device)]).float()
    return torch.nn.functional.cross_entropy(logits, y[1:][pos].to(logits.device))


@torch.no_grad()
def held_out_loss():
    model.eval()
    vals = [loss_of(i, lab).item() for i, lab in held]
    model.train()
    return round(sum(vals) / len(vals), 4) if vals else None


before = held_out_loss()
print("held-out loss before", before)

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
per_epoch = max(1, -(-len(train) // ACCUM))
steps = per_epoch * EPOCHS
save_steps = {max(1, round(e * per_epoch)): e for e in SAVE_AT}
warm = max(1, steps // 20)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, 1 - s / steps)
)
scaler = torch.amp.GradScaler("cuda")
model.train()
t0, step, running, curve, saved = time.time(), 0, [], [], []
for _epoch in range(EPOCHS):
    for n, (ids, labels) in enumerate(train, 1):
        loss = loss_of(ids, labels) / ACCUM
        scaler.scale(loss).backward()
        running.append(loss.item() * ACCUM)
        if n % ACCUM == 0 or n == len(train):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            step += 1
            if step % 5 == 0 or step == steps:
                mean = sum(running) / len(running)
                curve.append([step, round(mean, 4)])
                print(f"step {step}/{steps} loss {mean:.4f} {time.time() - t0:.0f}s", flush=True)
                running = []
            if step in save_steps:
                name = f"epoch-{save_steps[step]}"
                model.save_pretrained(OUT / name)
                saved.append({"name": name, "step": step, "held_out_loss": held_out_loss()})
                print("saved", saved[-1], flush=True)
summary = {
    "model": MODEL_ID,
    "conversations": len(rows),
    "train": len(train),
    "held_out": len(held),
    "skipped": skipped,
    "steps": steps,
    "lr": LR,
    "lora_r": LORA_R,
    "held_out_loss_before": before,
    "checkpoints": saved,
    "loss_curve": curve,
    "seconds": round(time.time() - t0),
}
Path("/kaggle/working/train.json").write_text(json.dumps(summary, indent=1))
print(json.dumps({k: v for k, v in summary.items() if k != "loss_curve"}, indent=1))